# CBMF4761 Project: Nonlinear Embedding Robustness & ADMIXTURE Validation
# Parker et al. (2017) Dog SNP Data

**Core questions:**

1. **Algorithm comparison** — Which nonlinear embedding method (UMAP, PaCMAP, t-SNE, TriMAP, PHATE) most robustly resolves dog breed population structure from 150K SNP genotypes?
2. **Parameter sensitivity** — How stable are each method's embeddings across large hyperparameter sweeps (~4150 total configs)?
3. **Biological validation** — Do embedding neighborhoods correspond to ADMIXTURE ancestry estimates (an independent, model-based method)?

**Data:** Parker et al. (2017), 1,346 dogs across 161 breeds, Illumina 170K CanineHD array; reference grouping uses 23 Parker clades (+ singletons + wolf outgroup).

**Methods:** PLINK2 for QC + LD pruning, 5-method embedding sweeps with trustworthiness / silhouette / seed stability metrics, ADMIXTURE across available K (`2..15, 23..25`) with deterministic elbow + interpretability diagnostics, concordance via ARI / NMI / neighborhood enrichment.


In [1]:
import os
import warnings
warnings.filterwarnings('ignore', message='IProgress')
warnings.filterwarnings('ignore', message='SGD-MDS')

from pathlib import Path
import pandas as pd
from helpers import lib

lib.NUM_CORES = os.cpu_count()
print(f'Using {lib.NUM_CORES} cores')


Using 12 cores


## Setup

Deterministic, cache-first workflow:
- ADMIXTURE was fit across the full continuous range `K = 2..30` (29 values); this notebook is cache-first and consumes the saved Q matrices in `results/admixture/`.
- Embedding sweeps load cached CSVs and only backfill missing runs.
- All downstream analysis uses the same frozen sample order, metadata, LD-pruned SNP matrix, and random seed.

In [2]:
ROOT = lib.find_repo_root(Path.cwd())
DATA_DIR   = ROOT / 'data'
RAW_PREFIX = DATA_DIR / 'All_Pure_150k'
RESULT_DIR = ROOT / 'results'
QC_DIR     = RESULT_DIR / 'qc'
QC_PREFIX  = QC_DIR / 'all_qc'
PRUNED_PREFIX = QC_DIR / 'all_qc_ldpruned'
RAW_MATRIX = QC_DIR / 'all_qc_ldpruned_additive.raw'
EMBED_DIR  = RESULT_DIR / 'embeddings'
FIG_DIR    = RESULT_DIR / 'figures'
METRIC_DIR = RESULT_DIR / 'metrics'
ADMIX_DIR  = RESULT_DIR / 'admixture'
CACHE_DIR  = RESULT_DIR / 'cache'
CLADE_CSV  = DATA_DIR / 'parker_clades.csv'

for d in [QC_DIR, EMBED_DIR, FIG_DIR, METRIC_DIR, ADMIX_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Canonical outputs
RUN_MANIFEST_PATH = METRIC_DIR / 'run_manifest.json'
ADMI_METRICS_PATH = METRIC_DIR / 'admixture_metrics_by_k.tsv'
EMBED_METRICS_PATH = METRIC_DIR / 'embedding_metrics_by_k.tsv'
PARKER_COMP_PATH = METRIC_DIR / 'parker_comparison.tsv'
SAMPLE_ORDER_PATH = METRIC_DIR / 'sample_order_master.tsv'
K_INTERPRET_PATH = ADMIX_DIR / 'k_interpretability.csv'
CV_DIAG_PATH = FIG_DIR / 'cv_diagnostics.png'
DIST_HEATMAP_PATH = FIG_DIR / 'distance_heatmap_ordered.png'
NJ_TREE_PATH = FIG_DIR / 'nj_tree.png'
DASHBOARD_PATH = FIG_DIR / 'comparison_dashboard.png'

# QC thresholds
GENO, MIND, MAF = 0.02, 0.02, 0.01
LD_WINDOW, LD_STEP, LD_R2 = 50, 5, 0.5

# UMAP sweep grid
UMAP_NEIGHBORS = [5, 10, 15, 30, 50, 100, 200]
UMAP_MIN_DISTS = [0.0, 0.1, 0.25, 0.5]
UMAP_METRICS   = ['euclidean', 'cosine', 'correlation', 'manhattan']

# PaCMAP sweep grid
PACMAP_NEIGHBORS = [5, 10, 15, 30, 50, 100, 200]
PACMAP_MN_RATIOS = [0.5, 1.0, 2.0]
PACMAP_FP_RATIOS = [1.0, 2.0, 4.0, 8.0]
PACMAP_DISTANCES = ['euclidean', 'angular', 'manhattan']

# t-SNE sweep grid
TSNE_PERPLEXITIES = [5, 10, 15, 30, 50, 100]
TSNE_METRICS      = ['euclidean', 'cosine', 'correlation', 'manhattan']

# TriMAP sweep grid
TRIMAP_N_INLIERS    = [5, 10, 15, 30, 50, 75, 100]
TRIMAP_N_OUTLIERS   = [4, 8, 15, 20, 30]
TRIMAP_DISTANCES    = ['euclidean', 'angular', 'manhattan']
TRIMAP_WEIGHT_TEMPS = [0.1, 0.25, 0.5]

# PHATE sweep grid
PHATE_KNN    = [5, 10, 15, 30, 50]
PHATE_DECAYS = [1, 5, 10, 40, 100]
PHATE_DISTS  = ['euclidean', 'cosine', 'manhattan']
PHATE_T      = ['auto', 2, 5]
PHATE_GAMMAS = [0, 1]

# Phylo-Autoencoder sweep grid
PHYLOAE_HIDDEN_DIMS   = [(512, 64), (1024, 128)]
PHYLOAE_LRS           = [1e-3, 3e-4]
PHYLOAE_LAMBDA_BREEDS = [0.1, 1.0, 10.0]
PHYLOAE_LAMBDA_CLADES = [0.1, 1.0]

# Shared
N_COMPONENTS = [2, 3]
SEEDS = [7, 42, 99, 123, 256]
ANALYSIS_SEED = 7
NJ_BOOTSTRAPS = 100

# ADMIXTURE cache inventory (fixed for this run)
ADMIX_K_RANGE = list(range(2, 31))
ADMIX_RUN_PREFIX = 'admixture_input'
ADMIX_ELBOW_DELTA = 0.002
ADMIXTURE_READ_ONLY_CACHE = True


# Never recompute sweeps in this notebook path; consume cached summaries + embeddings.
SKIP_SWEEP = True


### Parker assets

Two small derived files (`data/parker_clades.csv` and `data/parker_tree/parker_2017_nj.nex`)
come from the Parker 2017 Cell Reports paper. If absent locally they are downloaded
from the project's release mirror; see `data/README.md` for sources.

In [3]:
from helpers.data_assets import ensure_parker_assets
CLADE_CSV, PARKER_NJ = ensure_parker_assets(DATA_DIR)
print(f'clades: {CLADE_CSV}')
print(f'nj tree: {PARKER_NJ}')

clades: <repo>/data/parker_clades.csv
nj tree: <repo>/data/parker_tree/parker_2017_nj.nex


### Environment checks

Verifies `plink2` and `admixture` are installed and callable. On macOS Apple Silicon, ADMIXTURE runs under Rosetta 2 (x86_64 emulation). Confirms raw PLINK input files exist.

In [4]:
tools = lib.check_environment(ROOT, RAW_PREFIX)

Complete.


## Preprocessing

Prepares the raw 150K SNP data for embedding and ancestry analysis:

1. **Sample & variant QC** — Remove variants missing in >2% of samples (`--geno 0.02`), samples missing >2% of variants (`--mind 0.02`), and rare variants below 1% MAF.
2. **LD pruning** — Identify SNPs in approximate linkage equilibrium (window=50, step=5, r^2<0.5).
3. **Autosome filter + pruned subset** — Keep only autosomal, LD-pruned SNPs.
4. **Additive export** — Write the pruned genotype matrix as a `.raw` file (0/1/2 dosage per SNP).

In [5]:
required_outputs = [
    QC_PREFIX.with_suffix('.bed'), QC_PREFIX.with_suffix('.bim'), QC_PREFIX.with_suffix('.fam'),
    Path(str(QC_PREFIX) + '.prune.in'),
    PRUNED_PREFIX.with_suffix('.bed'), PRUNED_PREFIX.with_suffix('.bim'), PRUNED_PREFIX.with_suffix('.fam'),
    RAW_MATRIX,
]

if all(p.exists() for p in required_outputs):
    print('Complete.')
else:
    lib.run_plink_preprocessing(
        tools.plink2, RAW_PREFIX, QC_PREFIX, PRUNED_PREFIX, RAW_MATRIX,
        geno=GENO, mind=MIND, maf=MAF,
        ld_window=LD_WINDOW, ld_step=LD_STEP, ld_r2=LD_R2,
    )

meta, X = lib.load_plink_raw(RAW_MATRIX)
(meta, X_scaled, breed_order, color_mpl, color_plotly,
 clade_order, clade_color_mpl, clade_color_plotly) = lib.prepare_features(meta, X, clade_csv=CLADE_CSV)
print(
    f'Complete. {X_scaled.shape[0]} samples, {X_scaled.shape[1]} SNPs, '
    f'{len(breed_order)} breeds, 23 Parker clades (+ singletons + wolf outgroup)'
)


Complete.


Complete. 1337 samples, 63285 SNPs, 175 breeds, 23 Parker clades (+ singletons + wolf outgroup)


## Embedding sweeps

All six methods (UMAP, PaCMAP, t-SNE, TriMAP, PHATE, PhyloAE) share the same scaled SNP matrix; no breed or clade labels are passed to the unsupervised baselines. `SKIP_SWEEP=True` reads cached sweep summaries from `results/metrics/*.csv` rather than recomputing (the full 11,770-config sweep is reproducible via `helpers/sweep/run_sweep.py`).

### UMAP

UMAP (Uniform Manifold Approximation and Projection) builds a fuzzy k-NN graph in high-D and optimizes a low-D layout that preserves its topology. Excels at resolving tight local clusters; less emphasis on global inter-cluster distances. Supports diverse distance metrics natively.

**Swept parameters:**
- `n_neighbors` — Size of local neighborhood. Low = fine local detail, high = smoother global layout.
- `min_dist` — Minimum distance between points in the embedding. Low = tighter clusters, high = more spread.
- `metric` — Distance function in high-D space. Cosine/correlation often outperform euclidean for high-dimensional genetic data.

**Evaluation metrics** (shared across all methods):
- **Trustworthiness** — Do low-D neighbors match high-D neighbors? Near 1.0 = faithful local structure.
- **Silhouette by breed** — How well separated are breed clusters? Higher = tighter, more distinct.
- **Seed stability** — KNN overlap across random seeds with same hyperparameters. Higher = more reproducible.

In [6]:
theme = lib.apply_dark_theme()

if SKIP_SWEEP:
    umap_summary_path = METRIC_DIR / 'umap_sweep_summary.csv'
    if not umap_summary_path.exists():
        raise FileNotFoundError(f'Missing cached sweep summary: {umap_summary_path}')
    umap_summary = pd.read_csv(umap_summary_path)
    umap_embs = {}
    print(f'Loaded cached UMAP summary: {len(umap_summary)} rows')
else:
    umap_embs, umap_summary = lib.run_umap_sweep(
        X_scaled, meta,
        neighbors=UMAP_NEIGHBORS, min_dists=UMAP_MIN_DISTS, metrics=UMAP_METRICS,
        n_components=N_COMPONENTS, seeds=SEEDS,
        embed_dir=EMBED_DIR, metric_dir=METRIC_DIR,
    )
lib.summarize_sweep(umap_summary, ['metric', 'n_neighbors', 'min_dist', 'n_components'])


Loaded cached UMAP summary: 1120 rows
                    trust   silhou   stabil
metric
     correlation   0.8998   0.6844   0.7444
          cosine   0.8998   0.6858   0.7442
       euclidean   0.8137   0.2487   0.5794
       manhattan   0.8547   0.2793   0.6443
n_neighbors
               5   0.8331   0.7935   0.6512
              10   0.8651   0.7322   0.7161
              15   0.9101   0.5561   0.7589
              30   0.9030   0.4241   0.7231
              50   0.8806   0.3540   0.6765
             100   0.8519   0.2676   0.6290
             200   0.8252   0.1945   0.5917
min_dist
             0.0   0.8701   0.5123   0.6893
             0.1   0.8703   0.5009   0.6863
            0.25   0.8668   0.4698   0.6753
             0.5   0.8608   0.4153   0.6614
n_components
               2   0.8597   0.4469   0.6572
               3   0.8743   0.5022   0.6989


### PaCMAP

PaCMAP (Pairwise Controlled Manifold Approximation) optimizes in three phases: (1) pull nearby points together, (2) arrange mid-range pairs for global layout, (3) push far pairs apart. Designed to preserve inter-cluster distances better than UMAP, but lacks native support for many distance metrics (only euclidean, angular, manhattan).

**Swept parameters:**
- `n_neighbors` — Number of nearest neighbors for the attractive phase.
- `MN_ratio` — Ratio of mid-near pairs to near pairs. Higher = more global structure emphasis.
- `FP_ratio` — Ratio of far pairs to near pairs. Higher = stronger repulsion between distant points.
- `distance` — High-D distance metric (euclidean, angular/cosine, manhattan).

In [7]:
if SKIP_SWEEP:
    pacmap_summary_path = METRIC_DIR / 'pacmap_sweep_summary.csv'
    if not pacmap_summary_path.exists():
        raise FileNotFoundError(f'Missing cached sweep summary: {pacmap_summary_path}')
    pacmap_summary = pd.read_csv(pacmap_summary_path)
    pacmap_embs = {}
    print(f'Loaded cached PaCMAP summary: {len(pacmap_summary)} rows')
else:
    pacmap_embs, pacmap_summary = lib.run_pacmap_sweep(
        X_scaled, meta,
        neighbors=PACMAP_NEIGHBORS, mn_ratios=PACMAP_MN_RATIOS, fp_ratios=PACMAP_FP_RATIOS,
        distances=PACMAP_DISTANCES, n_components=N_COMPONENTS, seeds=SEEDS,
        embed_dir=EMBED_DIR, metric_dir=METRIC_DIR,
    )
lib.summarize_sweep(pacmap_summary, ['distance', 'n_neighbors', 'MN_ratio', 'FP_ratio', 'n_components'])


Loaded cached PaCMAP summary: 2520 rows
                    trust   silhou   stabil
distance
         angular   0.8676   0.4214   0.6357
       euclidean   0.8493   0.4203   0.6066
       manhattan   0.8510   0.4278   0.6333
n_neighbors
               5   0.8186   0.8087   0.6051
              10   0.8756   0.7257   0.6924
              15   0.8924   0.5822   0.6934
              30   0.8821   0.4126   0.6598
              50   0.8651   0.3199   0.6278
             100   0.8454   0.1660   0.5807
             200   0.8126  -0.0528   0.5175
MN_ratio
             0.5   0.8558   0.4213   0.6249
             1.0   0.8559   0.4230   0.6253
             2.0   0.8561   0.4252   0.6256
FP_ratio
             1.0   0.8254   0.2790   0.5653
             2.0   0.8498   0.3711   0.6092
             4.0   0.8688   0.4714   0.6499
             8.0   0.8798   0.5711   0.6766
n_components
               2   0.8453   0.3813   0.6019
               3   0.8667   0.4650   0.6486


### t-SNE

t-SNE (t-distributed Stochastic Neighbor Embedding) is the classic nonlinear embedding. It converts pairwise distances to conditional probabilities and minimizes KL divergence between high-D and low-D distributions. Excellent at revealing local cluster structure but notoriously slow, non-deterministic in global layout, and sensitive to perplexity.

**Swept parameters:**
- `perplexity` — Effective number of neighbors. Analogous to UMAP's `n_neighbors`. Low = local focus, high = broader context. Must be < n_samples.
- `metric` — Distance function in high-D space (euclidean, cosine, correlation, manhattan).

In [8]:
if SKIP_SWEEP:
    tsne_summary_path = METRIC_DIR / 'tsne_sweep_summary.csv'
    if not tsne_summary_path.exists():
        raise FileNotFoundError(f'Missing cached sweep summary: {tsne_summary_path}')
    tsne_summary = pd.read_csv(tsne_summary_path)
    tsne_embs = {}
    print(f'Loaded cached t-SNE summary: {len(tsne_summary)} rows')
else:
    tsne_embs, tsne_summary = lib.run_tsne_sweep(
        X_scaled, meta,
        perplexities=TSNE_PERPLEXITIES, metrics=TSNE_METRICS,
        n_components=N_COMPONENTS, seeds=SEEDS,
        embed_dir=EMBED_DIR, metric_dir=METRIC_DIR,
    )
lib.summarize_sweep(tsne_summary, ['metric', 'perplexity', 'n_components'])


Loaded cached t-SNE summary: 240 rows
                    trust   silhou   stabil
metric
     correlation   0.9143   0.7071   0.8439
          cosine   0.9128   0.7072   0.8443
       euclidean   0.8890   0.7092   0.7568
       manhattan   0.9137   0.7165   0.8034
perplexity
               5   0.8742   0.7629   0.7744
              10   0.9232   0.8004   0.8886
              15   0.9294   0.7760   0.9111
              30   0.9222   0.7029   0.8678
              50   0.9051   0.6271   0.7478
             100   0.8905   0.5909   0.6828
n_components
               2   0.9116   0.7825   0.8500
               3   0.9033   0.6376   0.7742


### TriMAP

TriMAP uses triplet constraints: for each point, a nearby "inlier" should be closer than a distant "outlier" in the embedding. This approach directly optimizes relative distance ordering rather than absolute positions. Claims better global structure preservation than t-SNE. No random_state parameter — inherently non-deterministic (seed stability reflects real variance).

**Swept parameters:**
- `n_inliers` — Number of nearest neighbors used as attractive anchors.
- `n_outliers` — Number of random distant points used as repulsive anchors per triplet.
- `distance` — High-D distance metric (euclidean, angular/cosine, manhattan).

In [9]:
if SKIP_SWEEP:
    trimap_summary_path = METRIC_DIR / 'trimap_sweep_summary.csv'
    if not trimap_summary_path.exists():
        raise FileNotFoundError(f'Missing cached sweep summary: {trimap_summary_path}')
    trimap_summary = pd.read_csv(trimap_summary_path)
    trimap_embs = {}
    print(f'Loaded cached TriMAP summary: {len(trimap_summary)} rows')
else:
    trimap_embs, trimap_summary = lib.run_trimap_sweep(
        X_scaled, meta,
        n_inliers_list=TRIMAP_N_INLIERS, n_outliers_list=TRIMAP_N_OUTLIERS,
        distances=TRIMAP_DISTANCES, weight_temps=TRIMAP_WEIGHT_TEMPS,
        n_components=N_COMPONENTS, seeds=SEEDS,
        embed_dir=EMBED_DIR, metric_dir=METRIC_DIR,
    )
lib.summarize_sweep(trimap_summary, ['distance', 'n_inliers', 'n_outliers', 'weight_temp', 'n_components'])


Loaded cached TriMAP summary: 3150 rows
                    trust   silhou   stabil
distance
         angular   0.7057  -0.1777   0.4297
       euclidean   0.7116  -0.1435   0.4861
       manhattan   0.6951  -0.1736   0.4930
n_inliers
               5   0.6562  -0.3005   0.3239
              10   0.6808  -0.2372   0.3581
              15   0.6939  -0.2006   0.3939
              30   0.7124  -0.1456   0.4725
              50   0.7226  -0.1111   0.5347
              75   0.7295  -0.0870   0.5848
             100   0.7335  -0.0724   0.6193
n_outliers
               4   0.6736  -0.2529   0.3567
               8   0.6947  -0.1941   0.4175
              15   0.7109  -0.1469   0.4866
              20   0.7168  -0.1278   0.5201
              30   0.7246  -0.1028   0.5671
weight_temp
             0.1   0.6996  -0.1717   0.4710
            0.25   0.7034  -0.1639   0.4702
             0.5   0.7094  -0.1591   0.4676
n_components
               2   0.6809  -0.2294   0.4354
               3   0.7274

### PHATE

PHATE (Potential of Heat-diffusion for Affinity-based Transition Embedding) was designed for biological data with trajectory/continuum structure. It builds a diffusion operator on the k-NN graph and embeds based on potential distances. The decay parameter controls how sharply the affinity kernel drops off, and diffusion time controls how far information propagates.

**Swept parameters:**
- `knn` — Number of nearest neighbors for the affinity graph.
- `decay` — Alpha-decay bandwidth. Low = sharp kernel (local focus), high = broad kernel (global diffusion).
- `knn_dist` — Distance metric for the k-NN graph (euclidean, cosine, manhattan).

In [10]:
if SKIP_SWEEP:
    phate_summary_path = METRIC_DIR / 'phate_sweep_summary.csv'
    if not phate_summary_path.exists():
        raise FileNotFoundError(f'Missing cached sweep summary: {phate_summary_path}')
    phate_summary = pd.read_csv(phate_summary_path)
    phate_embs = {}
    print(f'Loaded cached PHATE summary: {len(phate_summary)} rows')
else:
    phate_embs, phate_summary = lib.run_phate_sweep(
        X_scaled, meta,
        knn_values=PHATE_KNN, decays=PHATE_DECAYS, knn_dists=PHATE_DISTS,
        t_values=PHATE_T, gammas=PHATE_GAMMAS,
        n_components=N_COMPONENTS, seeds=SEEDS,
        embed_dir=EMBED_DIR, metric_dir=METRIC_DIR,
    )
lib.summarize_sweep(phate_summary, ['knn_dist', 'knn', 'decay', 't', 'gamma', 'n_components'])


Loaded cached PHATE summary: 4500 rows
                    trust   silhou   stabil
knn_dist
          cosine   0.7393  -0.0278   0.4715
       euclidean   0.7190  -0.0295   0.5350
       manhattan   0.6658  -0.1446   0.4691
knn
               5   0.5862  -0.2964   0.3334
              10   0.7331  -0.0084   0.5058
              15   0.7574   0.0185   0.5478
              30   0.7304  -0.0306   0.5314
              50   0.7332  -0.0196   0.5410
decay
               1   0.6319  -0.2966   0.4601
               5   0.6659  -0.1797   0.4599
              10   0.7107  -0.0606   0.4803
              40   0.7688   0.1075   0.5356
             100   0.7629   0.0930   0.5235
t
               2   0.6987  -0.0971   0.4741
               5   0.7059  -0.0601   0.4925
            auto   0.7196  -0.0447   0.5090
gamma
               0   0.7066  -0.1043   0.4746
               1   0.7095  -0.0302   0.5092
n_components
               2   0.6971  -0.1143   0.4779
               3   0.7190  -0.0203   0.50

### PhyloAE

A phylogeny-aware autoencoder that compresses 63K SNPs through a 2D/3D bottleneck while enforcing
multi-scale MDS-style constraints: breed centroids in embedding space should match PCA-derived
breed distances, and clade centroids should match clade distances. This is the only method that
incorporates known phylogenetic structure from Parker et al. 2017.

Loss = reconstruction + lambda_breed * breed_stress + lambda_clade * clade_stress

In [11]:
SKIP_PHYLOAE_SWEEP = os.environ.get('SKIP_PHYLOAE_SWEEP', '0') == '1'
PHYLOAE_TARGET_METRICS = ['euclidean', 'manhattan', 'cosine']

if SKIP_SWEEP or SKIP_PHYLOAE_SWEEP:
    phyloae_summary_path = METRIC_DIR / 'phyloae_sweep_summary.csv'
    if not phyloae_summary_path.exists():
        raise FileNotFoundError(f'Missing cached sweep summary: {phyloae_summary_path}')
    phyloae_summary = pd.read_csv(phyloae_summary_path)
    phyloae_embs = {}
    print(f'Loaded cached PhyloAE summary: {len(phyloae_summary)} rows')
else:
    phyloae_embs, phyloae_summary = lib.run_phyloae_sweep(
        X_scaled, meta,
        hidden_dims_list=PHYLOAE_HIDDEN_DIMS, lrs=PHYLOAE_LRS,
        lambda_breeds=PHYLOAE_LAMBDA_BREEDS, lambda_clades=PHYLOAE_LAMBDA_CLADES,
        n_components=N_COMPONENTS, seeds=[42, 123, 256, 789, 1337],
        embed_dir=EMBED_DIR, metric_dir=METRIC_DIR,
        target_metrics=PHYLOAE_TARGET_METRICS,
    )
lib.summarize_sweep(phyloae_summary, ['target_metric', 'hidden_dims', 'lr', 'lambda_breed', 'lambda_clade', 'n_components'])


Loaded cached PhyloAE summary: 720 rows
                    trust   silhou   stabil
target_metric
          cosine   0.9020   0.7603   0.6510
       euclidean   0.8887   0.7215   0.6612
       manhattan   0.8846   0.7237   0.6515
hidden_dims
     (1024, 128)   0.8910   0.7429   0.6518
       (512, 64)   0.8925   0.7275   0.6573
lr
          0.0003   0.8929   0.7278   0.6647
           0.001   0.8907   0.7426   0.6444
lambda_breed
             0.1   0.9034   0.7508   0.6624
             1.0   0.8889   0.7412   0.6496
            10.0   0.8829   0.7135   0.6516
lambda_clade
             0.1   0.8907   0.7369   0.6506
             1.0   0.8928   0.7335   0.6584
n_components
               2   0.8804   0.7157   0.6384
               3   0.9031   0.7546   0.6707


## Best-config selection

Generates canonical figures from each method's best setting (composite-ranked):

- **5-method comparison grid** — All methods side-by-side in a 2x3 panel.
- **Standalone 2D** per method with full breed legends.
- **3D interactive** HTML for each method via Plotly.

In [12]:
import hashlib
import json


def _best_by_composite(summary, *, n_components=None):
    sub = summary.copy()
    if n_components is not None:
        sub = sub[sub['n_components'] == n_components].copy()
    if sub.empty:
        raise ValueError(f'No rows available for n_components={n_components}')
    sub['composite'] = sub['trustworthiness'] + sub['silhouette_breed']
    sub = sub.dropna(subset=['composite'])
    if sub.empty:
        raise ValueError(f'No valid composite rows for n_components={n_components}')
    return sub.sort_values('composite', ascending=False).iloc[0].to_dict()


# Select best 2D, best 3D, and overall best-dim configs per method
methods = {}
for name, summary in [
    ('UMAP', umap_summary),
    ('PaCMAP', pacmap_summary),
    ('t-SNE', tsne_summary),
    ('TriMAP', trimap_summary),
    ('PHATE', phate_summary),
    ('PhyloAE', phyloae_summary),
]:
    best_2d = _best_by_composite(summary, n_components=2)
    best_3d = _best_by_composite(summary, n_components=3)
    best_overall = _best_by_composite(summary)
    print(
        f"Best {name}: dims={int(best_overall['n_components'])}, "
        f"trust={best_overall['trustworthiness']:.4f}, "
        f"silhou={best_overall['silhouette_breed']:.4f}, "
        f"stabil={best_overall['seed_stability_vs_base']:.4f}"
    )
    methods[name] = {
        'summary': summary,
        'best_2d': best_2d,
        'best_3d': best_3d,
        'best_overall': best_overall,
    }

BEST_CONFIG_PAYLOAD = {
    name: {
        'best_2d': {k: str(v) for k, v in methods[name]['best_2d'].items()},
        'best_3d': {k: str(v) for k, v in methods[name]['best_3d'].items()},
        'best_overall': {k: str(v) for k, v in methods[name]['best_overall'].items()},
    }
    for name in sorted(methods)
}
BEST_CONFIG_HASH = hashlib.sha1(
    json.dumps(BEST_CONFIG_PAYLOAD, sort_keys=True).encode('utf-8')
).hexdigest()[:12]
print('Best-config hash:', BEST_CONFIG_HASH)

prefix_map = {
    'UMAP': 'UMAP',
    'PaCMAP': 'PaCMAP',
    't-SNE': 'TSNE',
    'TriMAP': 'TriMAP',
    'PHATE': 'PHATE',
    'PhyloAE': 'PhyloAE',
}


def _load_best_embedding(best_row, method_name, n_components):
    csv_path = Path(str(best_row['embedding_csv']))
    if not csv_path.exists():
        raise FileNotFoundError(f'Missing cached best embedding for {method_name}: {csv_path}')
    return lib._load_embedding_from_csv(csv_path, prefix_map[method_name], n_components)


emb_2d = {name: _load_best_embedding(methods[name]['best_2d'], name, 2) for name in methods}
emb_3d = {name: _load_best_embedding(methods[name]['best_3d'], name, 3) for name in methods}
emb_best = {
    name: _load_best_embedding(
        methods[name]['best_overall'],
        name,
        int(methods[name]['best_overall']['n_components']),
    )
    for name in methods
}
dimensions_used = {name: int(methods[name]['best_overall']['n_components']) for name in methods}
print('Best-dim map:', dimensions_used)

summary_inputs = [
    METRIC_DIR / 'umap_sweep_summary.csv',
    METRIC_DIR / 'pacmap_sweep_summary.csv',
    METRIC_DIR / 'tsne_sweep_summary.csv',
    METRIC_DIR / 'trimap_sweep_summary.csv',
    METRIC_DIR / 'phate_sweep_summary.csv',
    METRIC_DIR / 'phyloae_sweep_summary.csv',
]

comparison_grid_path = FIG_DIR / 'all_methods_2d_comparison.png'
if lib.outputs_up_to_date([comparison_grid_path], summary_inputs):
    print('Comparison grid cached:', comparison_grid_path)
else:
    panels = [(emb_2d[n], n) for n in methods]
    lib.plot_comparison_grid(panels, meta, breed_order, color_mpl, FIG_DIR, theme,
                             clade_order=clade_order, clade_color_mpl=clade_color_mpl)

for name in methods:
    prefix = name.replace('-', '')
    two_d_path = FIG_DIR / f"{name.lower()}_2d_breed.png"
    three_d_path = FIG_DIR / f"{name.lower()}_3d_interactive.html"
    best_2d_csv = Path(str(methods[name]['best_2d'].get('embedding_csv', '')))
    best_3d_csv = Path(str(methods[name]['best_3d'].get('embedding_csv', '')))

    if lib.outputs_up_to_date([two_d_path], [best_2d_csv] if best_2d_csv.exists() else summary_inputs):
        print(f'{name} 2D cached:', two_d_path)
    else:
        lib.plot_embedding_2d(
            emb_2d[name], meta, breed_order, color_mpl,
            method_name=name, method_params=methods[name]['best_2d'],
            axis_prefix=prefix, fig_dir=FIG_DIR, theme=theme,
            clade_order=clade_order, clade_color_mpl=clade_color_mpl,
        )

    if lib.outputs_up_to_date([three_d_path], [best_3d_csv] if best_3d_csv.exists() else summary_inputs):
        print(f'{name} 3D cached:', three_d_path)
    else:
        lib.plot_3d_interactive(
            emb_3d[name], meta, breed_order, color_plotly, FIG_DIR, theme,
            method_name=name, axis_prefix=prefix,
            clade_order=clade_order, clade_color_plotly=clade_color_plotly,
        )


Best UMAP: dims=3, trust=0.8695, silhou=0.8181, stabil=1.0000
Best PaCMAP: dims=3, trust=0.8327, silhou=0.8557, stabil=0.5069
Best t-SNE: dims=2, trust=0.9318, silhou=0.8406, stabil=0.8559
Best TriMAP: dims=3, trust=0.7756, silhou=0.0690, stabil=0.7041
Best PHATE: dims=3, trust=0.8968, silhou=0.6271, stabil=0.6265
Best PhyloAE: dims=3, trust=0.9224, silhou=0.8352, stabil=0.6070
Best-config hash: f0cfb36ae2d9
Best-dim map: {'UMAP': 3, 'PaCMAP': 3, 't-SNE': 2, 'TriMAP': 3, 'PHATE': 3, 'PhyloAE': 3}
Comparison grid cached: <repo>/results/figures/all_methods_2d_comparison.png
UMAP 2D cached: <repo>/results/figures/umap_2d_breed.png
UMAP 3D cached: <repo>/results/figures/umap_3d_interactive.html
PaCMAP 2D cached: <repo>/results/figures/pacmap_2d_breed.png
PaCMAP 3D cached: <repo>/results/figures/pacmap_3d_interactive.html
t-SNE 2D cached: <repo>/results/figures/t-sne_2d_breed.png
t-SNE 3D cached: <repo>/results/figures/t-sne_3d_interactive.html
TriMAP 2D cached: <repo>/results/figures/trima

## External #1 — ADMIXTURE concordance (label-free)

ADMIXTURE was fit across the full continuous range **K = 2..30** (29 values). All Q matrices live in `results/admixture/`; this cell is cache-first — it consumes those matrices if present, otherwise re-runs the ADMIXTURE binary for the missing Ks.

Pipeline:
1. canonical CV table
2. K selection (elbow heuristic on the K=2..15 sub-range, where the curve is still informative)
3. master sample order
4. cross-K component alignment (Hungarian on 1−correlation)
5. per-K ADMIXTURE metrics (ARI / NMI vs each embedding's clusters)
6. standardized barplots

K-selection rule:
- `delta_cv(K) = CV(K-1) − CV(K)` for sequential K
- elbow on K=2..15: smallest K with `delta_cv < 0.002`
- fallback 1: smallest K after which all remaining `delta_cv` are below threshold
- fallback 2: maximize `delta2_cv = delta_cv(K) − delta_cv(K+1)`

Setting `ADMIXTURE_READ_ONLY_CACHE = True` (in the configuration cell) flips the cache-miss behavior from "run ADMIXTURE" to "raise an error" — useful when the cache should be authoritative.

In [13]:
import hashlib
import json
import numpy as np
import pandas as pd

method_summaries = {
    'UMAP': umap_summary,
    'PaCMAP': pacmap_summary,
    't-SNE': tsne_summary,
    'TriMAP': trimap_summary,
    'PHATE': phate_summary,
    'PhyloAE': phyloae_summary,
}
method_summary_paths = {
    'UMAP': METRIC_DIR / 'umap_sweep_summary.csv',
    'PaCMAP': METRIC_DIR / 'pacmap_sweep_summary.csv',
    't-SNE': METRIC_DIR / 'tsne_sweep_summary.csv',
    'TriMAP': METRIC_DIR / 'trimap_sweep_summary.csv',
    'PHATE': METRIC_DIR / 'phate_sweep_summary.csv',
    'PhyloAE': METRIC_DIR / 'phyloae_sweep_summary.csv',
}

manifest = lib.build_run_manifest(
    ROOT, RAW_PREFIX, PRUNED_PREFIX, RAW_MATRIX,
    ADMIX_DIR, ADMIX_RUN_PREFIX, ADMIX_K_RANGE, SEEDS,
    RUN_MANIFEST_PATH, method_summary_paths=method_summary_paths,
)

preflight = lib.preflight_closed_form(
    PRUNED_PREFIX, RAW_MATRIX, ADMIX_DIR, ADMIX_RUN_PREFIX,
    ADMIX_K_RANGE, len(meta), method_summaries,
)
print('Preflight checks passed:', preflight)

available_q = sorted(k for k in ADMIX_K_RANGE if (ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k}.Q').exists())
missing_q = sorted(set(ADMIX_K_RANGE) - set(available_q))
if missing_q and ADMIXTURE_READ_ONLY_CACHE:
    raise RuntimeError(f'ADMIXTURE_READ_ONLY_CACHE=True but Q matrices are missing for K={missing_q}. Either set the flag to False to run ADMIXTURE, or restore the cache.')

q_input_paths = [ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k}.Q' for k in available_q]
log_input_paths = [ADMIX_DIR / f'log_k{k}.out' for k in available_q]

cv_all_path = ADMIX_DIR / 'cv_errors_all.tsv'
if lib.outputs_up_to_date([cv_all_path], q_input_paths + log_input_paths):
    cv_df = pd.read_csv(cv_all_path, sep='	')
else:
    cv_df = lib.build_admixture_cv_table(ADMIX_DIR, ADMIX_RUN_PREFIX)

k_summary_path = ADMIX_DIR / 'k_selection_summary.json'
if lib.outputs_up_to_date([k_summary_path], [cv_all_path]):
    k_selection = json.loads(k_summary_path.read_text(encoding='utf-8'))
else:
    k_selection = lib.select_admixture_k_values(cv_df, elbow_threshold=ADMIX_ELBOW_DELTA)
    ld_status = lib.assert_ld_pruned_input(PRUNED_PREFIX)
    lib.write_k_selection_summary(ADMIX_DIR, k_selection, ld_status)
ld_status = lib.assert_ld_pruned_input(PRUNED_PREFIX)

interpret_inputs = [ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k}.Q' for k in range(2, 16)]
if lib.outputs_up_to_date([K_INTERPRET_PATH], interpret_inputs):
    interpret_df = pd.read_csv(K_INTERPRET_PATH)
else:
    interpret_df = lib.build_k_interpretability(
        ADMIX_DIR, ADMIX_RUN_PREFIX, n_samples=len(meta), k_min=2, k_max=15,
    )

k_cv_min = int(k_selection['k_cv_min'])
k_elbow = int(k_selection['k_elbow'])
k_broad = int(k_selection['k_broad_default'])
k_fine = int(k_selection['k_fine_default'])
print('Selected Ks:', {'k_cv_min': k_cv_min, 'k_elbow': k_elbow, 'k_broad_default': k_broad, 'k_fine_default': k_fine})
print('Interpretability diagnostics saved:', K_INTERPRET_PATH)

q_by_k = lib.load_q_matrices(ADMIX_DIR, ADMIX_RUN_PREFIX, available_q, len(meta))

alignment_key_payload = {
    'run_prefix': ADMIX_RUN_PREFIX,
    'available_q': available_q,
}
alignment_key = hashlib.sha1(json.dumps(alignment_key_payload, sort_keys=True).encode('utf-8')).hexdigest()[:12]
ALIGNMENT_CACHE_PATH = CACHE_DIR / f'admixture_alignment_{alignment_key}.npz'

alignment_force = not lib.outputs_up_to_date([ALIGNMENT_CACHE_PATH], q_input_paths)

def _restore_int_keys(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            key = int(k) if isinstance(k, str) and k.isdigit() else k
            out[key] = _restore_int_keys(v)
        return out
    if isinstance(obj, list):
        return [_restore_int_keys(v) for v in obj]
    return obj

def _compute_alignment_cache():
    aligned = lib.align_components_across_k(q_by_k, available_q)
    return {'alignment_json': np.array([json.dumps(aligned)])}

alignment_payload = lib.load_or_compute(
    ALIGNMENT_CACHE_PATH,
    _compute_alignment_cache,
    force=alignment_force,
    fmt='npz',
)
alignment = _restore_int_keys(json.loads(str(alignment_payload['alignment_json'][0])))

if lib.outputs_up_to_date([CV_DIAG_PATH], [ADMIX_DIR / 'cv_errors.tsv', ADMIX_DIR / 'k_selection_summary.json']):
    print('CV diagnostics cached:', CV_DIAG_PATH)
else:
    lib.plot_cv_diagnostics_standard(cv_df, k_selection, CV_DIAG_PATH, theme)

best_method, broad_conc_df = lib.choose_best_concordant_embedding(emb_best, q_by_k[k_broad], seed=ANALYSIS_SEED)

sample_order_inputs = [
    Path(str(methods[best_method]['best_overall'].get('embedding_csv', ''))),
    ADMIX_DIR / 'k_selection_summary.json',
]
if lib.outputs_up_to_date([SAMPLE_ORDER_PATH], sample_order_inputs):
    sample_order_df = pd.read_csv(SAMPLE_ORDER_PATH, sep='	')
    sample_order_idx = sample_order_df['sample_idx'].to_numpy(dtype=int)
else:
    sample_order_df, sample_order_idx = lib.build_master_sample_order(
        meta, q_by_k[k_broad], emb_best[best_method], best_method, SAMPLE_ORDER_PATH,
    )

admi_base_key_payload = {
    'best_config_hash': BEST_CONFIG_HASH,
    'available_q': available_q,
    'k_selection': {k: k_selection[k] for k in sorted(k_selection)},
}
ADMI_METRICS_BASE_CACHE_PATH = METRIC_DIR / f"admixture_metrics_base_{hashlib.sha1(json.dumps(admi_base_key_payload, sort_keys=True).encode('utf-8')).hexdigest()[:12]}.tsv"

admi_base_inputs = [ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k}.Q' for k in available_q]
admi_base_inputs += [ADMIX_DIR / f'log_k{k}.out' for k in available_q]
admi_base_force = not lib.outputs_up_to_date([ADMI_METRICS_BASE_CACHE_PATH], admi_base_inputs)

def _compute_admixture_metrics_base():
    return lib.compute_admixture_metrics_by_k(
        q_by_k, cv_df, k_selection, alignment, ADMI_METRICS_BASE_CACHE_PATH,
    )

admixture_metrics_df = lib.load_or_compute(
    ADMI_METRICS_BASE_CACHE_PATH,
    _compute_admixture_metrics_base,
    force=admi_base_force,
    fmt='tsv',
)

barplot_paths = [FIG_DIR / f'barplot_K{k}.png' for k in sorted(q_by_k.keys())]
barplot_paths.append(FIG_DIR / 'barplot_panel_broad_fine.png')
barplot_inputs = [SAMPLE_ORDER_PATH] + [ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k}.Q' for k in sorted(q_by_k.keys())]
if lib.outputs_up_to_date(barplot_paths, barplot_inputs):
    barplot_outputs = {'per_k': [p for p in barplot_paths if p.name.startswith('barplot_K')], 'panel': FIG_DIR / 'barplot_panel_broad_fine.png'}
    print('ADMIXTURE barplots cached:', len(barplot_outputs['per_k']))
else:
    barplot_outputs = lib.plot_admixture_barplots_master_order(
        q_by_k, sample_order_idx, alignment, FIG_DIR, theme, broad_k=k_broad, fine_k=k_fine,
    )

print('Best concordant embedding for master order:', best_method)
print('Broad concordance table:')
print(broad_conc_df.to_string(index=False))
print('Barplots generated:', len(barplot_outputs['per_k']))


Preflight checks passed: {'k_inventory': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30], 'n_samples_expected': 1337, 'summary_rows': {'UMAP': 1120, 'PaCMAP': 2520, 't-SNE': 240, 'TriMAP': 3150, 'PHATE': 4500, 'PhyloAE': 720}, 'admixture_cache_ok': True, 'embedding_cache_ok': True}
Selected Ks: {'k_cv_min': 30, 'k_elbow': 10, 'k_broad_default': 8, 'k_fine_default': 15}
Interpretability diagnostics saved: <repo>/results/admixture/k_interpretability.csv


CV diagnostics cached: <repo>/results/figures/cv_diagnostics.png


ADMIXTURE barplots cached: 29
Best concordant embedding for master order: PHATE
Broad concordance table:
 method      ari      nmi
  PHATE 0.095196 0.371672
  t-SNE 0.086169 0.366720
PhyloAE 0.099953 0.357328
 TriMAP 0.073436 0.280525
   UMAP 0.030925 0.260287
 PaCMAP 0.021273 0.173539
Barplots generated: 29


### Embedding × ADMIXTURE concordance

This stage computes:
- embedding ARI/NMI, local purity, logistic separability (broad/fine K)
- fixed-name embedding figures by dominant component and entropy
- K ranking, component ranking, ordered distance heatmap
- bootstrapped NJ tree (breed + clade)
- deterministic Parker comparison classification and summary dashboard

In [14]:
import hashlib
import json
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA

target_ks = sorted(set([k_broad, k_fine]))

embed_key_payload = {
    'best_config_hash': BEST_CONFIG_HASH,
    'target_ks': target_ks,
    'best_method': best_method,
}
embed_key = hashlib.sha1(json.dumps(embed_key_payload, sort_keys=True).encode('utf-8')).hexdigest()[:12]
EMBED_METRICS_CACHE_PATH = METRIC_DIR / f'embedding_metrics_by_k_{embed_key}.tsv'

embed_inputs = [Path(str(methods[name]['best_overall'].get('embedding_csv', ''))) for name in sorted(methods)]
embed_inputs += [ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k}.Q' for k in target_ks]
embed_force = not lib.outputs_up_to_date([EMBED_METRICS_CACHE_PATH], embed_inputs)

def _compute_embedding_metrics():
    return lib.compute_embedding_metrics_and_figures(
        emb_best, meta, q_by_k, target_ks, alignment,
        EMBED_METRICS_CACHE_PATH, FIG_DIR, theme, seed=ANALYSIS_SEED,
    )

embedding_metrics_df = lib.load_or_compute(
    EMBED_METRICS_CACHE_PATH,
    _compute_embedding_metrics,
    force=embed_force,
    fmt='tsv',
)
embedding_metrics_df.to_csv(EMBED_METRICS_PATH, sep='	', index=False)

embedding_fig_paths = []
for method in emb_best:
    slug = ''.join(ch for ch in method.lower() if ch.isalnum())
    for k in target_ks:
        embedding_fig_paths.append(FIG_DIR / f'embedding_{slug}_by_dominantK{k}.png')
        embedding_fig_paths.append(FIG_DIR / f'embedding_{slug}_by_entropyK{k}.png')
if not lib.outputs_up_to_date(embedding_fig_paths, [EMBED_METRICS_CACHE_PATH]):
    print('Embedding figures stale; regenerating from cached inputs...')
    embedding_metrics_df = _compute_embedding_metrics()
    embedding_metrics_df.to_csv(EMBED_METRICS_CACHE_PATH, sep='	', index=False)
    embedding_metrics_df.to_csv(EMBED_METRICS_PATH, sep='	', index=False)

rank_key_payload = {
    'best_config_hash': BEST_CONFIG_HASH,
    'best_method': best_method,
    'k_broad': k_broad,
    'k_fine': k_fine,
}
rank_key = hashlib.sha1(json.dumps(rank_key_payload, sort_keys=True).encode('utf-8')).hexdigest()[:12]
ADMI_METRICS_RANKED_CACHE_PATH = METRIC_DIR / f'admixture_metrics_ranked_{rank_key}.tsv'

rank_force = not lib.outputs_up_to_date(
    [ADMI_METRICS_RANKED_CACHE_PATH],
    [ADMI_METRICS_BASE_CACHE_PATH, EMBED_METRICS_CACHE_PATH],
)

def _compute_ranked_metrics():
    ranked = lib.rank_k_candidates(admixture_metrics_df.copy(), embedding_metrics_df, ADMI_METRICS_RANKED_CACHE_PATH)
    ranked = lib.apply_component_ranking(
        ranked, q_by_k, emb_best[best_method], ADMI_METRICS_RANKED_CACHE_PATH, seed=ANALYSIS_SEED,
    )
    return ranked

admixture_metrics_df = lib.load_or_compute(
    ADMI_METRICS_RANKED_CACHE_PATH,
    _compute_ranked_metrics,
    force=rank_force,
    fmt='tsv',
)
admixture_metrics_df.to_csv(ADMI_METRICS_PATH, sep='	', index=False)

dominant_broad = q_by_k[k_broad].argmax(axis=1).astype(int)
DIST_HEATMAP_CACHE_PATH = CACHE_DIR / f'distance_heatmap_{BEST_CONFIG_HASH}_k{k_broad}.npz'

heatmap_inputs = [
    RAW_MATRIX,
    SAMPLE_ORDER_PATH,
    Path(str(methods[best_method]['best_overall'].get('embedding_csv', ''))),
]
heatmap_force = not lib.outputs_up_to_date([DIST_HEATMAP_CACHE_PATH], heatmap_inputs)

def _compute_distance_heatmap_cache():
    n_comp = int(min(max(2, 20), X_scaled.shape[0] - 1, X_scaled.shape[1]))
    pca = PCA(n_components=n_comp, random_state=ANALYSIS_SEED)
    Xp = pca.fit_transform(X_scaled)
    D = squareform(pdist(Xp, metric='euclidean'))
    D = D[np.ix_(sample_order_idx, sample_order_idx)]

    clade = meta['breed_group'].astype(str).to_numpy()[sample_order_idx]
    clade_vals = pd.factorize(clade)[0].astype(np.int32)
    dom = dominant_broad[sample_order_idx].astype(np.int32)
    dom_rgb = np.array([mcolors.to_rgb(alignment['colors_by_k'][k_broad][int(c)]) for c in dom], dtype=np.float64)
    dom_rgb = dom_rgb.reshape(1, len(dom), 3)

    return {
        'D': D.astype(np.float32),
        'clade_vals': clade_vals,
        'dom_rgb': dom_rgb,
    }

heatmap_cache = lib.load_or_compute(
    DIST_HEATMAP_CACHE_PATH,
    _compute_distance_heatmap_cache,
    force=heatmap_force,
    fmt='npz',
)

if not lib.outputs_up_to_date([DIST_HEATMAP_PATH], [DIST_HEATMAP_CACHE_PATH]):
    t = theme
    D = heatmap_cache['D']
    clade_vals = heatmap_cache['clade_vals']
    dom_rgb = heatmap_cache['dom_rgb']

    fig, axes = plt.subplots(
        3, 1, figsize=(10.8, 10.6), dpi=180,
        gridspec_kw={'height_ratios': [0.18, 0.18, 9.64]},
        facecolor=t['fig_bg'],
    )
    for ax in axes:
        ax.set_facecolor(t['ax_bg'])
    axes[0].imshow(dom_rgb, aspect='auto')
    axes[0].set_ylabel('ADMIX', fontsize=7)
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    axes[1].imshow(clade_vals.reshape(1, -1), aspect='auto', cmap=plt.get_cmap('tab20b'))
    axes[1].set_ylabel('Meta', fontsize=7)
    axes[1].set_xticks([])
    axes[1].set_yticks([])

    im = axes[2].imshow(D, aspect='auto', cmap='viridis')
    axes[2].set_title('Distance heatmap (ordered by fixed master sample order)', fontsize=10)
    axes[2].set_xlabel('Sample order rank')
    axes[2].set_ylabel('Sample order rank')
    cbar = fig.colorbar(im, ax=axes[2], fraction=0.030, pad=0.01)
    cbar.ax.tick_params(labelsize=7, colors=t['text'])
    fig.tight_layout()
    fig.savefig(DIST_HEATMAP_PATH, bbox_inches='tight', facecolor=t['fig_bg'])
    plt.close(fig)
else:
    print('Distance heatmap cached:', DIST_HEATMAP_PATH)

NJ_TREE_CACHE_PATH = CACHE_DIR / f'nj_tree_bootstrap_stats_b{NJ_BOOTSTRAPS}_s{ANALYSIS_SEED}.npz'
tree_inputs = [RAW_MATRIX, CLADE_CSV]
tree_force = (
    not lib.outputs_up_to_date([NJ_TREE_CACHE_PATH], tree_inputs)
    or not lib.outputs_up_to_date([NJ_TREE_PATH], tree_inputs)
)

def _tree_stats_to_npz_payload(stats):
    payload = {}
    for group in ['breed', 'clade']:
        for key, value in stats[group].items():
            arr_key = f'{group}__{key}'
            if isinstance(value, (int, np.integer, bool)):
                payload[arr_key] = np.array([int(value)], dtype=np.int64)
            elif isinstance(value, (float, np.floating)):
                payload[arr_key] = np.array([float(value)], dtype=np.float64)
            else:
                payload[arr_key] = np.array([str(value)])
    return payload

def _tree_stats_from_npz_payload(payload):
    stats = {'breed': {}, 'clade': {}}
    for full_key, value in payload.items():
        group, key = full_key.split('__', 1)
        scalar = value[0]
        if isinstance(scalar, np.generic):
            scalar = scalar.item()
        stats[group][key] = scalar
    return stats

def _compute_tree_cache():
    stats = lib.build_nj_tree_figure(
        X_scaled, meta, NJ_TREE_PATH, theme,
        n_bootstrap=NJ_BOOTSTRAPS, seed=ANALYSIS_SEED,
    )
    return _tree_stats_to_npz_payload(stats)

tree_cache_payload = lib.load_or_compute(
    NJ_TREE_CACHE_PATH,
    _compute_tree_cache,
    force=tree_force,
    fmt='npz',
)
tree_stats = _tree_stats_from_npz_payload(tree_cache_payload)

parker_key_payload = {
    'best_config_hash': BEST_CONFIG_HASH,
    'k_broad': k_broad,
    'k_fine': k_fine,
    'tree_cache': NJ_TREE_CACHE_PATH.name,
}
parker_key = hashlib.sha1(json.dumps(parker_key_payload, sort_keys=True).encode('utf-8')).hexdigest()[:12]
PARKER_COMP_CACHE_PATH = METRIC_DIR / f'parker_comparison_{parker_key}.tsv'

parker_force = not lib.outputs_up_to_date(
    [PARKER_COMP_CACHE_PATH],
    [ADMI_METRICS_RANKED_CACHE_PATH, EMBED_METRICS_CACHE_PATH, NJ_TREE_CACHE_PATH],
)

def _compute_parker_table():
    return lib.build_parker_comparison_table(
        admixture_metrics_df, embedding_metrics_df, tree_stats,
        k_selection, PARKER_COMP_CACHE_PATH,
    )

parker_df = lib.load_or_compute(
    PARKER_COMP_CACHE_PATH,
    _compute_parker_table,
    force=parker_force,
    fmt='tsv',
)
parker_df.to_csv(PARKER_COMP_PATH, sep='	', index=False)

if lib.outputs_up_to_date([DASHBOARD_PATH], [ADMI_METRICS_PATH, EMBED_METRICS_PATH, PARKER_COMP_PATH, CV_DIAG_PATH]):
    print('Comparison dashboard cached:', DASHBOARD_PATH)
else:
    lib.plot_comparison_dashboard(
        cv_df, k_selection, admixture_metrics_df, embedding_metrics_df,
        parker_df, DASHBOARD_PATH, theme,
    )

CONCORDANCE_BROAD_CACHE_PATH = METRIC_DIR / f'concordance_broad_{BEST_CONFIG_HASH}_k{k_broad}.tsv'
conc_force = not lib.outputs_up_to_date(
    [CONCORDANCE_BROAD_CACHE_PATH],
    [Path(str(methods[name]['best_overall'].get('embedding_csv', ''))) for name in sorted(methods)] +
    [ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k_broad}.Q'],
)

def _compute_broad_concordance_df():
    rows = []
    q_b = q_by_k[k_broad]
    lbl_b = q_b.argmax(axis=1)
    for name in methods:
        conc = lib.compute_concordance(emb_best[name], q_b, lbl_b, k_broad, len(meta), name, seed=ANALYSIS_SEED)
        rows.append(conc)
    return pd.DataFrame(rows)

concordance_df = lib.load_or_compute(
    CONCORDANCE_BROAD_CACHE_PATH,
    _compute_broad_concordance_df,
    force=conc_force,
    fmt='tsv',
)

concordances = []
for _, row in concordance_df.iterrows():
    conc = {
        'method': str(row['method']),
        'best_k': int(row['best_k']),
        'ari': float(row['ari']),
        'nmi': float(row['nmi']),
        'neighbor_mean_l1_q': float(row['neighbor_mean_l1_q']),
        'random_mean_l1_q': float(row['random_mean_l1_q']),
        'neighbor_enrichment_ratio': float(row['neighbor_enrichment_ratio']),
    }
    concordances.append(conc)
    methods[conc['method']]['concordance'] = conc

admixture_barplot_path = FIG_DIR / 'admixture_barplot_best_k.png'
if lib.outputs_up_to_date([admixture_barplot_path], [CONCORDANCE_BROAD_CACHE_PATH, ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k_broad}.Q']):
    print('ADMIXTURE barplot cached:', admixture_barplot_path)
else:
    q_values, admix_label, component_colors, component_names = lib.load_q_matrix(
        ADMIX_DIR, ADMIX_RUN_PREFIX, k_broad, len(meta)
    )
    lib.plot_admixture_barplot(
        q_values, emb_best[best_method], k_broad, component_colors, component_names,
        concordances, FIG_DIR, theme,
    )

print('Tree stats:', tree_stats)
print('Parker comparison:')
print(parker_df.to_string(index=False))
print('Embedding metrics preview:')
print(embedding_metrics_df.head(6).to_string(index=False))
print('Ranked admixture metrics preview:')
print(admixture_metrics_df.head(8).to_string(index=False))


Embedding figures stale; regenerating from cached inputs...


Distance heatmap cached: <repo>/results/figures/distance_heatmap_ordered.png
ADMIXTURE barplot cached: <repo>/results/figures/admixture_barplot_best_k.png
Tree stats: {'breed': {'label_name': 'breed', 'n_groups': 175, 'n_internal_splits': 172, 'n_support_ge_50': 146, 'n_support_ge_70': 127, 'n_support_ge_90': 107, 'pct_support_ge_70': 0.7383720930232558}, 'clade': {'label_name': 'clade', 'n_groups': 26, 'n_internal_splits': 23, 'n_support_ge_50': 22, 'n_support_ge_70': 19, 'n_support_ge_90': 17, 'pct_support_ge_70': 0.8260869565217391}}
Parker comparison:
                   feature                                     parker_pattern                                                                                                                                                                                                                   our_value comparison_flag
         tree_discreteness  supported internal clades in bootstrapped NJ tree                                         {"labe

### Concordance across all K + cluster estimation

Compute ARI/NMI at every K from 2-30 for all 6 methods, and estimate the natural number of clusters via silhouette sweep (K=2-50).

In [15]:
import hashlib
import json
from pathlib import Path

# Concordance across all K values
conc_key_payload = {
    'best_config_hash': BEST_CONFIG_HASH,
    'available_q': sorted(q_by_k.keys()),
}
conc_key = hashlib.sha1(json.dumps(conc_key_payload, sort_keys=True).encode('utf-8')).hexdigest()[:12]
CONC_ALL_K_CACHE_PATH = METRIC_DIR / f'concordance_all_k_{conc_key}.tsv'

conc_inputs = [Path(str(methods[name]['best_overall'].get('embedding_csv', ''))) for name in sorted(methods)]
conc_inputs += [ADMIX_DIR / f'{ADMIX_RUN_PREFIX}.{k}.Q' for k in sorted(q_by_k.keys())]
conc_force = not lib.outputs_up_to_date([CONC_ALL_K_CACHE_PATH], conc_inputs)

def _compute_concordance_all_k_df():
    return lib.compute_concordance_all_k(emb_best, q_by_k, seed=ANALYSIS_SEED)

conc_all_k_df = lib.load_or_compute(
    CONC_ALL_K_CACHE_PATH,
    _compute_concordance_all_k_df,
    force=conc_force,
    fmt='tsv',
)
conc_all_k_df.to_csv(METRIC_DIR / 'concordance_all_k.tsv', sep='	', index=False)
if lib.outputs_up_to_date([FIG_DIR / 'concordance_vs_k.png'], [CONC_ALL_K_CACHE_PATH]):
    print('Concordance-vs-K figure cached:', FIG_DIR / 'concordance_vs_k.png')
else:
    lib.plot_concordance_vs_k(conc_all_k_df, FIG_DIR / 'concordance_vs_k.png', theme)

# Cluster count estimation
cluster_key_payload = {
    'best_config_hash': BEST_CONFIG_HASH,
    'k_range': [2, 50],
}
cluster_key = hashlib.sha1(json.dumps(cluster_key_payload, sort_keys=True).encode('utf-8')).hexdigest()[:12]
CLUSTER_CACHE_PATH = METRIC_DIR / f'cluster_estimation_{cluster_key}.tsv'

cluster_inputs = [Path(str(methods[name]['best_overall'].get('embedding_csv', ''))) for name in sorted(methods)]
cluster_force = not lib.outputs_up_to_date([CLUSTER_CACHE_PATH], cluster_inputs)

def _compute_cluster_df():
    return lib.estimate_cluster_count(emb_best, k_range=range(2, 51), seed=ANALYSIS_SEED)

cluster_df = lib.load_or_compute(
    CLUSTER_CACHE_PATH,
    _compute_cluster_df,
    force=cluster_force,
    fmt='tsv',
)
cluster_df.to_csv(METRIC_DIR / 'cluster_estimation.tsv', sep='	', index=False)
if lib.outputs_up_to_date([FIG_DIR / 'cluster_estimation.png'], [CLUSTER_CACHE_PATH]):
    print('Cluster-estimation figure cached:', FIG_DIR / 'cluster_estimation.png')
else:
    lib.plot_cluster_estimation(cluster_df, FIG_DIR / 'cluster_estimation.png', theme, parker_k=26)

# Report peak K per method
for method in sorted(emb_best.keys()):
    sub = cluster_df[cluster_df['method'] == method]
    peak = sub.loc[sub['silhouette'].idxmax()]
    print(f'{method}: peak silhouette at K={int(peak["K"])} ({peak["silhouette"]:.3f})')


Concordance-vs-K figure cached: <repo>/results/figures/concordance_vs_k.png
Cluster-estimation figure cached: <repo>/results/figures/cluster_estimation.png
PHATE: peak silhouette at K=50 (0.602)
PaCMAP: peak silhouette at K=50 (0.510)
PhyloAE: peak silhouette at K=2 (0.534)
TriMAP: peak silhouette at K=2 (0.999)
UMAP: peak silhouette at K=50 (0.588)
t-SNE: peak silhouette at K=49 (0.555)


## External #2 — Parker-tree validation

External validation against Parker et al. (2017)'s original neighbor-joining tree
(supplemental file mmc5, committed locally as `data/parker_tree/parker_2017_nj.nex`
with citation and provenance in `data/parker_tree/README.md`).

Two analyses run here, both cache-first:

1. **Mantel correlation** - each method's breed-centroid pairwise distances are
   correlated against Parker's patristic distances at four combos
   (Euclidean x {Pearson, Spearman}, Manhattan x {Pearson, Spearman}).
   Driver: `scripts/mantel_nj.py --reference parker` -> `results/metrics/mantel_nj.json`
   and `results/figures/presentation/mantel_nj.png`.

2. **Novel-breed clade support** - for each of the 6 consensus-misplaced breeds
   (Airedale Terrier, Silky Terrier, Yorkshire Terrier, Weimaraner, Dalmatian,
   Great Dane), all 26 Parker clades are ranked by mean patristic distance from
   the breed's tips, and the assigned clade's rank k/26 is reported (raw,
   Hungarian-inclusive - the rank cited on the final slide).
   Driver: `scripts/novel_breed_parker_support.py` -> `results/metrics/novel_breed_parker_support.json`.

Cache-first: each cell below skips rerunning the driver if its JSON already
exists, unless `FORCE_PARKER_RERUN = True` is set.


In [16]:
# Mantel correlation vs Parker's NJ tree (cache-first)
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

FORCE_PARKER_RERUN = False  # set True to re-execute drivers even if JSON exists

MANTEL_JSON = METRIC_DIR / 'mantel_nj.json'
MANTEL_FIG  = FIG_DIR / 'presentation' / 'mantel_nj.png'
PARKER_TREE = ROOT / 'data' / 'parker_tree' / 'parker_2017_nj.nex'

assert PARKER_TREE.exists(), f'Parker tree missing: {PARKER_TREE}'

if MANTEL_JSON.exists() and not FORCE_PARKER_RERUN:
    print(f'[cached] {MANTEL_JSON}')
else:
    print('Running helpers/figures/mantel_nj.py --reference parker ...')
    subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'mantel_nj.py'),
         '--reference', 'parker', '--parker-tree', str(PARKER_TREE)],
        check=True, cwd=ROOT,
    )

mantel = json.loads(MANTEL_JSON.read_text())
rows = []
for method, combos in mantel['per_method_mantel'].items():
    rows.append({
        'Method': method,
        'Euclid-Pearson':   combos['euclidean_pearson'],
        'Euclid-Spearman':  combos['euclidean_spearman'],
        'Manhattan-Pearson':   combos['manhattan_pearson'],
        'Manhattan-Spearman':  combos['manhattan_spearman'],
    })
mantel_df = (pd.DataFrame(rows)
               .sort_values('Euclid-Pearson', ascending=False)
               .reset_index(drop=True))
print(f"Reference: {mantel['reference']}")
print(f"Shared breeds per method: {mantel['shared_breed_counts']}")
display(mantel_df.style.format({c: '{:.3f}' for c in mantel_df.columns if c != 'Method'}))

# Slide-matching headline: PhyloAE lead at Euclid-Pearson vs t-SNE
phyloae_r = mantel['per_method_mantel']['PhyloAE']['euclidean_pearson']
tsne_r    = mantel['per_method_mantel']['t-SNE']['euclidean_pearson']
ratio = phyloae_r / tsne_r if tsne_r else float('nan')
print(f"\nPhyloAE leads at r = {phyloae_r:.3f} (Euclidean-Pearson); "
      f"{ratio:.2f}x t-SNE ({tsne_r:.3f}).")


[cached] <repo>/results/metrics/mantel_nj.json
Reference: parker_2017_patristic_breed_mean
Shared breeds per method: {'UMAP': 175, 'PaCMAP': 175, 't-SNE': 175, 'TriMAP': 175, 'PHATE': 175, 'PhyloAE': 175}


,Method,Euclid-Pearson,Euclid-Spearman,Manhattan-Pearson,Manhattan-Spearman
0,PhyloAE,0.588,0.571,0.584,0.573
1,t-SNE,0.490,0.475,0.485,0.468
2,UMAP,0.389,0.365,0.379,0.352
3,PHATE,0.371,0.396,0.408,0.412
4,PaCMAP,0.292,0.265,0.287,0.256
5,TriMAP,-0.002,0.283,-0.001,0.295



PhyloAE leads at r = 0.588 (Euclidean-Pearson); 1.20x t-SNE (0.490).


In [17]:
# Per-breed Parker NJ-tree support for 6 novel consensus-misplaced breeds (cache-first)
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

NOVEL_JSON = METRIC_DIR / 'novel_breed_parker_support.json'

if NOVEL_JSON.exists() and not FORCE_PARKER_RERUN:
    print(f'[cached] {NOVEL_JSON}')
else:
    print('Running helpers/figures/novel_breed_support.py ...')
    subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'novel_breed_parker_support.py')],
        check=True, cwd=ROOT,
    )

novel = json.loads(NOVEL_JSON.read_text())

def _verdict(code, v):
    rank = v['rank_of_assigned']
    if rank is None:
        return 'N/A'
    if rank == 1:
        return 'STRONG'
    if rank == 2:
        # Raw rank=2 is Hungarian-edged-out: clade is actually rank 1/25 ex-Hungarian.
        return 'STRONG (edged out only by Hungarian)'
    # rank >= 3 means the tree places the breed outside its assigned clade
    # Report the nearest NON-Hungarian clade as 'nearest', since the Hungarian
    # clade is most distance-central in Parker's tree (rank 1/26 inter-clade)
    # and dominates the top slot spuriously.
    nearest_non_hungarian = next(
        (e['clade'] for e in v['ranked_all'] if e['clade'] != 'Hungarian'),
        v['top5'][0]['clade'],
    )
    return f'TREE-OFF (nearest: {nearest_non_hungarian})'

order = ['AIRT', 'SILK', 'YORK', 'WEIM', 'DALM', 'DANE']
rows = []
for code in order:
    v = novel['tree_rankings'][code]
    rows.append({
        'breed':        v['breed_full'],
        'assigned':     v['assigned_clade'],
        'rank (of 26)': v['rank_of_assigned'],
        'verdict':      _verdict(code, v),
    })
novel_df = pd.DataFrame(rows)
print(f"Tree stats: {novel['tree_stats']['n_tips_total']} tips, "
      f"{novel['tree_stats']['n_clades_in_tree_mapping']} clades")
display(novel_df)

buckets = novel['summary_buckets']
print(f"\nBuckets (raw rank, Hungarian-inclusive):")
print(f"  STRONG (rank 1):    {buckets['STRONG'] or '[]'}")
print(f"  WEAK   (rank 2-3):  {buckets['WEAK']   or '[]'}")
print(f"  AGAINST (rank >=4): {buckets['AGAINST'] or '[]'}")


[cached] <repo>/results/metrics/novel_breed_parker_support.json
Tree stats: 1355 tips, 26 clades


,breed,assigned,rank (of 26),verdict
0,Airedale Terrier,Terrier,2,STRONG (edged out only by Hungarian)
1,Silky Terrier,Terrier,2,STRONG (edged out only by Hungarian)
2,Yorkshire Terrier,Terrier,2,STRONG (edged out only by Hungarian)
3,Weimaraner,PointerSetter,2,STRONG (edged out only by Hungarian)
4,Dalmatian,PointerSetter,11,TREE-OFF (nearest: NewWorld)
5,Great Dane,EuropeanMastiff,6,TREE-OFF (nearest: Alpine)



Buckets (raw rank, Hungarian-inclusive):
  STRONG (rank 1):    []
  WEAK   (rank 2-3):  ['Airedale Terrier', 'Silky Terrier', 'Yorkshire Terrier', 'Weimaraner']
  AGAINST (rank >=4): ['Dalmatian', 'Great Dane']


## Output validation

Verifies the closed-form artifacts (tables, figures) referenced by the report and slides are all present.

In [18]:
required_tables = [
    ADMIX_DIR / 'cv_errors_all.tsv',
    ADMIX_DIR / 'k_selection_summary.json',
    K_INTERPRET_PATH,
    ADMI_METRICS_PATH,
    EMBED_METRICS_PATH,
    PARKER_COMP_PATH,
    SAMPLE_ORDER_PATH,
]
required_figs = [
    CV_DIAG_PATH,
    FIG_DIR / 'barplot_panel_broad_fine.png',
    DIST_HEATMAP_PATH,
    NJ_TREE_PATH,
    DASHBOARD_PATH,
]
for k in sorted(q_by_k.keys()):
    required_figs.append(FIG_DIR / f'barplot_K{k}.png')
for m in emb_2d:
    slug = ''.join(ch for ch in m.lower() if ch.isalnum())
    for k in sorted(set([k_broad, k_fine])):
        required_figs.append(FIG_DIR / f'embedding_{slug}_by_dominantK{k}.png')
        required_figs.append(FIG_DIR / f'embedding_{slug}_by_entropyK{k}.png')

missing_tables = [str(p) for p in required_tables if not p.exists()]
missing_figs = [str(p) for p in required_figs if not p.exists()]
if missing_tables or missing_figs:
    raise RuntimeError(f'Missing outputs. tables={missing_tables}, figures={missing_figs}')

print('All required closed-form outputs generated successfully.')
print('Run manifest:', RUN_MANIFEST_PATH)
print('Primary outputs:')
for p in required_tables + required_figs[:8]:
    print(' -', p)

All required closed-form outputs generated successfully.
Run manifest: <repo>/results/metrics/run_manifest.json
Primary outputs:
 - <repo>/results/admixture/cv_errors_all.tsv
 - <repo>/results/admixture/k_selection_summary.json
 - <repo>/results/admixture/k_interpretability.csv
 - <repo>/results/metrics/admixture_metrics_by_k.tsv
 - <repo>/results/metrics/embedding_metrics_by_k.tsv
 - <repo>/results/metrics/parker_comparison.tsv
 - <repo>/results/metrics/sample_order_master.tsv
 - <repo>/results/figures/cv_diagnostics.png
 - <repo>/results/figures/barplot_panel_broad_fine.png
 - <repo>/results/figures/distance_heatmap_ordered.png
 - <repo>/results/figures/nj_tree.png
 - <repo>/results/figures/comparison_dashboard.png
 - <repo>/results/figures/barplot_K2.png
 - <repo>/results/figures/barplot_K3.png
 - <repo>/results/figures/barplot_K4.png
